In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import os
from dotenv import load_dotenv
import duckdb
load_dotenv()

True

In [2]:
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_USER")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_PASSWORD")
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute("SET s3_region='us-east-1';")

In [ ]:
df = con.execute("""
    SELECT avg(f.aqi), f.date
    FROM read_parquet(
        's3://stream-analytics-project-bucket/gold/airnow/fact_air_quality_readings/*/*/*.parquet'
    ) f
    JOIN read_parquet(
        's3://stream-analytics-project-bucket/gold/airnow/dim_parameter/*.parquet'
    ) p 
    ON f.parameter_key = p.parameter_key'
    WHERE f.date BETWEEN '2025-12-01' AND '2026-02-28' AND p.parameter = 'PM2.5' 
    GROUP BY f.date
""").df()

In [3]:
df = con.execute("""
    SELECT 
        s.sitename,
        AVG(f.aqi) AS avg_aqi
    FROM read_parquet(
        's3://stream-analytics-project-bucket/gold/airnow/fact_air_quality_readings/*/*/*.parquet'
    ) f
    JOIN read_parquet(
        's3://stream-analytics-project-bucket/gold/airnow/dim_site/*.parquet'
    ) s
    ON f.site_key = s.site_key
    GROUP BY s.sitename
    ORDER BY avg_aqi DESC
    LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.barh(df["sitename"], df["avg_aqi"])
plt.xlabel("Average AQI")
plt.ylabel("Site")
plt.title("Top 10 Most Polluted Sites")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# df.show()